In [0]:
import pandas as pd
import numpy as np
from collections import defaultdict, deque

predictions = spark.table("workspace.gold.worldcup_2026_match_predictions").toPandas()

predictions["date"] = pd.to_datetime(predictions["date"])

print("Partidos cargados:", predictions.shape)
display(predictions.head())

In [0]:
print(predictions.columns.tolist())

In [0]:
# Crear grafo de equipos conectados por partido
graph = defaultdict(set)

for _, row in predictions.iterrows():
    home = row["home_team"]
    away = row["away_team"]
    graph[home].add(away)
    graph[away].add(home)

# Buscar componentes conectados
visited = set()
groups = []

for team in graph:
    if team not in visited:
        queue = deque([team])
        visited.add(team)
        component = []

        while queue:
            current = queue.popleft()
            component.append(current)

            for neighbor in graph[current]:
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(neighbor)

        groups.append(sorted(component))

# Asignar nombres de grupos
group_names = [chr(ord("A") + i) for i in range(len(groups))]

group_map = {}

for group_name, teams in zip(group_names, groups):
    for team in teams:
        group_map[team] = group_name

predictions["group"] = predictions["home_team"].map(group_map)

print("Número de grupos encontrados:", len(groups))

for group_name, teams in zip(group_names, groups):
    print(group_name, teams)

In [0]:
def simulate_match(row):
    outcomes = ["home_win", "draw", "away_win"]
    probabilities = [
        row["prob_home_win"],
        row["prob_draw"],
        row["prob_away_win"]
    ]

    return np.random.choice(outcomes, p=probabilities)

In [0]:
def simulate_group_stage(predictions_df):
    tables = {}

    for group in sorted(predictions_df["group"].unique()):
        group_matches = predictions_df[predictions_df["group"] == group]

        teams = sorted(set(group_matches["home_team"]).union(set(group_matches["away_team"])))

        table = {
            team: {
                "team": team,
                "group": group,
                "points": 0,
                "wins": 0,
                "draws": 0,
                "losses": 0,
                "random_tiebreaker": np.random.random()
            }
            for team in teams
        }

        for _, row in group_matches.iterrows():
            result = simulate_match(row)

            home = row["home_team"]
            away = row["away_team"]

            if result == "home_win":
                table[home]["points"] += 3
                table[home]["wins"] += 1
                table[away]["losses"] += 1

            elif result == "away_win":
                table[away]["points"] += 3
                table[away]["wins"] += 1
                table[home]["losses"] += 1

            else:
                table[home]["points"] += 1
                table[away]["points"] += 1
                table[home]["draws"] += 1
                table[away]["draws"] += 1

        group_table = pd.DataFrame(table.values())

        group_table = group_table.sort_values(
            ["points", "wins", "random_tiebreaker"],
            ascending=[False, False, False]
        ).reset_index(drop=True)

        group_table["position"] = group_table.index + 1

        tables[group] = group_table

    return tables

In [0]:
def get_qualified_teams(group_tables):
    all_tables = pd.concat(group_tables.values(), ignore_index=True)

    top_two = all_tables[all_tables["position"] <= 2].copy()

    third_places = all_tables[all_tables["position"] == 3].copy()

    best_thirds = third_places.sort_values(
        ["points", "wins", "random_tiebreaker"],
        ascending=[False, False, False]
    ).head(8)

    qualified = pd.concat([top_two, best_thirds], ignore_index=True)

    return qualified

In [0]:
# Precalcular probabilidades de partidos ya existentes
match_prob_lookup = {}

for _, row in predictions.iterrows():
    home = row["home_team"]
    away = row["away_team"]
    
    match_prob_lookup[(home, away)] = (
        row["prob_home_win"],
        row["prob_draw"],
        row["prob_away_win"]
    )
    
    # También guardamos el partido invertido
    match_prob_lookup[(away, home)] = (
        row["prob_away_win"],
        row["prob_draw"],
        row["prob_home_win"]
    )

# Precalcular puntos FIFA por equipo
team_points = {}

for _, row in predictions.iterrows():
    team_points[row["home_team"]] = row["fifa_points_home"]
    team_points[row["away_team"]] = row["fifa_points_away"]


def get_match_probabilities_fast(team_a, team_b):
    # Si ya existe una probabilidad calculada para ese cruce, usarla
    if (team_a, team_b) in match_prob_lookup:
        return match_prob_lookup[(team_a, team_b)]
    
    # Si no existe el cruce, aproximar usando puntos FIFA
    points_a = team_points.get(team_a, 1500)
    points_b = team_points.get(team_b, 1500)
    
    diff = points_a - points_b
    
    prob_a_raw = 1 / (1 + np.exp(-diff / 300))
    prob_draw = 0.22
    
    prob_a = prob_a_raw * (1 - prob_draw)
    prob_b = 1 - prob_a - prob_draw
    
    return prob_a, prob_draw, prob_b


def simulate_knockout_match(team_a, team_b):
    prob_a, prob_draw, prob_b = get_match_probabilities_fast(team_a, team_b)

    result = np.random.choice(
        ["team_a", "draw", "team_b"],
        p=[prob_a, prob_draw, prob_b]
    )

    if result == "team_a":
        return team_a
    elif result == "team_b":
        return team_b
    else:
        return np.random.choice([team_a, team_b])

In [0]:
def simulate_worldcup_once(predictions_df):
    group_tables = simulate_group_stage(predictions_df)
    qualified = get_qualified_teams(group_tables)

    qualified = qualified.sort_values(
        ["points", "wins", "random_tiebreaker"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    teams = qualified["team"].tolist()

    round_32 = teams
    round_16 = []

    for i in range(16):
        team_a = round_32[i]
        team_b = round_32[-(i + 1)]
        winner = simulate_knockout_match(team_a, team_b)
        round_16.append(winner)

    quarterfinalists = []

    for i in range(0, len(round_16), 2):
        winner = simulate_knockout_match(round_16[i], round_16[i + 1])
        quarterfinalists.append(winner)

    semifinalists = []

    for i in range(0, len(quarterfinalists), 2):
        winner = simulate_knockout_match(quarterfinalists[i], quarterfinalists[i + 1])
        semifinalists.append(winner)

    finalists = []

    for i in range(0, len(semifinalists), 2):
        winner = simulate_knockout_match(semifinalists[i], semifinalists[i + 1])
        finalists.append(winner)

    champion = simulate_knockout_match(finalists[0], finalists[1])

    return {
        "qualified_round_32": round_32,
        "round_16": round_16,
        "quarterfinalists": quarterfinalists,
        "semifinalists": semifinalists,
        "finalists": finalists,
        "champion": champion
    }

In [0]:
N_SIMULATIONS = 5000

simulation_results = []

for i in range(N_SIMULATIONS):
    result = simulate_worldcup_once(predictions)
    simulation_results.append(result)

    if (i + 1) % 100 == 0:
        print(f"Simulaciones completadas: {i + 1}")

print("Simulaciones finalizadas.")

In [0]:
display(simulation_summary.sort_values("prob_champion", ascending=False).head(20))

In [0]:
all_teams = sorted(
    set(predictions["home_team"]).union(set(predictions["away_team"]))
)

summary = []

for team in all_teams:
    round_32_count = sum(team in sim["qualified_round_32"] for sim in simulation_results)
    round_16_count = sum(team in sim["round_16"] for sim in simulation_results)
    quarter_count = sum(team in sim["quarterfinalists"] for sim in simulation_results)
    semi_count = sum(team in sim["semifinalists"] for sim in simulation_results)
    final_count = sum(team in sim["finalists"] for sim in simulation_results)
    champion_count = sum(team == sim["champion"] for sim in simulation_results)

    summary.append({
        "team": team,
        "prob_round_32": round_32_count / N_SIMULATIONS,
        "prob_round_16": round_16_count / N_SIMULATIONS,
        "prob_quarterfinal": quarter_count / N_SIMULATIONS,
        "prob_semifinal": semi_count / N_SIMULATIONS,
        "prob_final": final_count / N_SIMULATIONS,
        "prob_champion": champion_count / N_SIMULATIONS
    })

simulation_summary = pd.DataFrame(summary)

simulation_summary = simulation_summary.sort_values(
    "prob_champion",
    ascending=False
)

display(simulation_summary.head(20))

In [0]:
simulation_summary_spark = spark.createDataFrame(simulation_summary)

simulation_summary_spark.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("workspace.gold.worldcup_2026_montecarlo_summary")

print("Resumen Monte Carlo guardado en Gold.")

In [0]:
team_fifa_info = {}

for _, row in predictions.iterrows():
    team_fifa_info[row["home_team"]] = {
        "fifa_rank": row["fifa_rank_home"],
        "fifa_points": row["fifa_points_home"]
    }
    team_fifa_info[row["away_team"]] = {
        "fifa_rank": row["fifa_rank_away"],
        "fifa_points": row["fifa_points_away"]
    }

simulation_summary["fifa_rank"] = simulation_summary["team"].map(
    lambda team: team_fifa_info.get(team, {}).get("fifa_rank")
)

simulation_summary["fifa_points"] = simulation_summary["team"].map(
    lambda team: team_fifa_info.get(team, {}).get("fifa_points")
)

simulation_summary = simulation_summary[
    [
        "team",
        "fifa_rank",
        "fifa_points",
        "prob_round_32",
        "prob_round_16",
        "prob_quarterfinal",
        "prob_semifinal",
        "prob_final",
        "prob_champion"
    ]
]

display(simulation_summary.sort_values("prob_champion", ascending=False).head(20))

In [0]:
predictions[
    (predictions["home_team"] == "Colombia") |
    (predictions["away_team"] == "Colombia")
][[
    "date",
    "home_team",
    "away_team",
    "fifa_rank_home",
    "fifa_rank_away",
    "fifa_points_home",
    "fifa_points_away"
]]